# Strings, files, and errors

Reading text that people wrote, and saying clearly when something is wrong.

## How we'll work today

As before: **Predict** before you run. Write your answer down, compare with your neighbour, then run the cell.

# Part 1: Taking text apart

## Recap: `split`

On day 1 we split a sentence into words. Exercise 3's lines looked like `4000 trip.jpg`. 

**Predict:** what does this print?

In [ ]:
line = "4000 trip.jpg"
print(line.split())
size, name = line.split()
print(size + size)
print(int(size) + int(size))

`split()` always gives **strings**, even when they look like numbers: `"4000" + "4000"` glues them together. `int(...)` turns the text into a number. Unpacking (`size, name = ...`) works because the list has exactly two elements.

## splitting on different characters

**Predict:** what does each line print?

In [ ]:
print(" a  b ".split())
print(" a  b ".split(" "))
print("1,2,,3".split(","))
print("".split(","))
print("".split())

Two different tools with the same name:

- `split()` with **nothing** in the brackets splits at any run of spaces, tabs or line breaks, and never gives you empty strings.
- `split(sep)` splits at **every** `sep`. Two `sep`s in a row, or one at the start or end, give an empty string `""` in between.

So `"".split(",")` is `[""]`, a list with one empty string, not an empty list. Whenever you split at a separator, ask yourself what happens to the empty pieces.

## Cleaning up the edges: `strip`

Text from a file or a person often has extra spaces, and a line break at the end. `repr(...)` shows a string the way you'd write it in code, so you can see what's really there.

**Predict:**

In [ ]:
s = "  42\n"
print(s)
print(repr(s))
print(repr(s.strip()))
print(int(s))

- `"\n"` is **one** character, a line break. `print` shows it as a new line; `repr` shows it as `\n`.
- `.strip()` removes spaces, tabs and line breaks at both ends (never in the middle). Like every string method, it returns a new string.
- `int(...)` forgives spaces around the number. But

**predict**:

In [ ]:
print(int(""))

A `ValueError`: there is no number in an empty string. Part 3 is about what to do with errors like this one.

## Lines

**Predict:** how many elements does each list have?

In [ ]:
text = "first\nsecond\n\nfourth\n"
print(text.splitlines())
print(text.split("\n"))

`.splitlines()` splits at line breaks and doesn't produce an extra `""` for the line break at the very end; `.split("\n")` does. The empty line in the middle is `""` in both.

## Everything before a character

Suppose a line can end with a remark after a `%`. **Predict:**

In [ ]:
print("Ada 17  % handed in late".split("%"))
print("Ada 17  % handed in late".split("%")[0])
print("Grace 20".split("%")[0])

`[0]` is the part before the first `%`, and if there's no `%` at all, it's the whole string. Either way you get something, so no special case is needed.

## Is it a number?

**Predict:**

In [ ]:
print("42".isdigit())
print("-42".isdigit())
print(" 42".isdigit())
print(int("-42"), int(" 42"))

`.isdigit()` asks "is every character a digit?" A minus sign or a space isn't one, although `int` handles both. Checking `isdigit()` before `int(...)` silently drops negative numbers.

## Characters from everywhere

A string can hold characters from any writing system: it's a sequence of **Unicode** characters. Each has a number (`ord`) and an official name, which the standard library module `unicodedata` knows.

In [ ]:
import unicodedata

for char in "a3²€😀":
    print(char, ord(char), unicodedata.name(char))

**Predict:**

In [ ]:
micro = "µ"
mu = "μ"
print(micro, mu)
# print(micro == mu)
# print(unicodedata.name(micro))
# print(unicodedata.name(mu))

`"\u00b5"` is "the character with number b5 (hexadecimal)". These two look the same and are different characters: µ is on a German keyboard (AltGr+m), μ is the Greek letter. Text that looks equal isn't necessarily equal.

**Predict:**

In [ ]:
cafe_1 = "caf\u00e9"
cafe_2 = "cafe\u0301"
print(cafe_1, cafe_2)
# print(cafe_1 == cafe_2)
# print(len(cafe_1), len(cafe_2))

In [ ]:
print([unicodedata.name(char) for char in cafe_1])
print([unicodedata.name(char) for char in cafe_2])

`cafe_2` writes é as two characters: `e`, then an accent that attaches to the character before it. Both forms are legal Unicode and both occur in practice (macOS, for example, stores file names in the second form). Before comparing text from different sources, bring it into one form:

In [ ]:
print(unicodedata.normalize("NFC", cafe_1) == unicodedata.normalize("NFC", cafe_2))

`"NFC"` joins letter and accent into one character wherever possible; `"NFD"` splits them.

## Digits from everywhere

Back to `isdigit()`. 



In [ ]:
print("²".isdigit())
print(int("²"))

`"²"` (superscript two, as in m²) counts as a digit for `isdigit()`, but `int` can't read it. So `isdigit()` is wrong in **both** directions as a test for "can `int` read this?".

**Predict:**

In [ ]:
print("٤٢".isdigit(), unicodedata.name("٤"), unicodedata.name("٢"))
print("४२".isdigit(), unicodedata.name("४"), unicodedata.name("२"))
print("፬፪".isdigit(), unicodedata.name("፬"), unicodedata.name("፪"))

In [ ]:
print(int("४२"))
print(int("٤٢"))
print(int("፬፪"))

`isdigit()` knows the digits of every script, and so does `int`: `٤٢` (Arabic-Indic) and `४२` (Devanagari) are both 42. Ethiopic numerals don't work like ours (there is no zero, nothing is written by position), so `int` can't read `፬፪`, although it is digits 4 and 2. `isdigit()` and `int` answer different questions.

## And `float`?

**Predict:** which lines work?

In [ ]:
print(float("3.5"), float(" -3.5 "))
print(float("1e3"), float("1_000"))
print(float("inf"), float("nan"))
print(int("3.5"))

`float` reads more than `int`: a decimal point, `e` for "times ten to the", and even `"inf"` and `"nan"` ("not a number"). `int` reads only whole numbers. Both allow a sign, spaces around the number and `_` between digits, like in Python code.

| text | `isdigit()` | `int(text)` | `float(text)` |
|---|---|---|---|
| `"42"`, `"٤٢"` | `True` | 42 | 42.0 |
| `"-42"`, `" 42 "`, `"+42"`, `"4_2"` | `False` | 42 or -42 | 42.0 or -42.0 |
| `"3.5"`, `"1e3"`, `"inf"`, `"nan"` | `False` | `ValueError` | works |
| `"²"`, `"፫"` | `True` | `ValueError` | `ValueError` |
| `""` | `False` | `ValueError` | `ValueError` |

There are also `.isdecimal()` (stricter) and `.isnumeric()` (looser: `"½"` and `"Ⅻ"` count). None of the three answers "will `int(...)` work?". Only `int(...)` itself does: Part 3 shows how to just try it, and handle the error if there is one.

# Part 2: Files

## Writing a file

In [ ]:
with open("shopping.txt", "w") as file:
    file.write("apples\n")
    file.write("pears\n")

- `open(name, "w")` opens a file for **w**riting; `open(name)` alone opens it for reading.
- `file.write(text)` writes exactly the text you give it, nothing more: no line break unless you write `"\n"` yourself.
- `with ... as file:` is a block like `for` or `if`. More on it in a moment.

## Reading a file

**Predict:** what does this print?

In [ ]:
with open("shopping.txt") as file:
    text = file.read()
print(repr(text))

`file.read()` gives the whole file as **one string**, line breaks included. Everything from Part 1 works on it.

You can also go through a file line by line with `for`.

**Predict:** what does this print?

In [ ]:
with open("shopping.txt") as file:
    for line in file:
        print(line)

Each `line` still ends with its `"\n"`, and `print` adds another one: blank lines in between. `line.strip()` gets rid of it.

## What `with` does

**Predict:** what do the two `print`s show?

In [ ]:
with open("shopping.txt") as file:
    text = file.read()
print(file.closed)
print(file.read())

When the `with` block ends, the file is **closed**, so nothing can be read from it anymore. Python closes it even if an error happens inside the block. Without `with`, you'd have to call `file.close()` yourself, and remember to do it on every path through your code, including when something goes wrong. Always open files with `with`.

So: read inside the block, keep what you need in a name (`text`), and work with that afterwards.

## `"w"` again

**Predict:** what's in the file afterwards?

In [ ]:
with open("shopping.txt", "w") as file:
    file.write("bread\n")

with open("shopping.txt") as file:
    print(file.read())

`"w"` empties the file first, without asking. To add to the end instead, open it with `"a"` (**a**ppend).

## Where is `"shopping.txt"`?

A bare file name is looked up in the **working directory**: the folder Python was started from, not the folder the code is in.

In [ ]:
from pathlib import Path

print(Path.cwd())

In this notebook that's `lectures/`. With `uv run python students/<login>/ex04/ex04.py`, it's the top of the repository, so `open("diagnostic.txt")` would look for `diagnostic.txt` there, not next to `ex04.py`.

`pathlib` (standard library) has `Path`, an object for a file's path. `/` joins parts of a path, whatever the operating system's separator.

**Predict:**

In [ ]:
p = Path("data") / "scores.txt"
print(p)
print(p.name)
print(p.suffix)
print(p.parent)
print(p.exists())

In a `.py` file, Python sets the name `__file__` to that file's own path. So

```python
PROGRAM_FILE = Path(__file__).parent / "diagnostic.txt"
```

is "`diagnostic.txt`, in the same folder as this file", wherever Python was started from. Exercise 4 gives you that line. (A notebook has no `__file__`, so we can't run it here.)

`open(...)` accepts a `Path` as well as a string. For a quick read, `Path` also has a shortcut that opens, reads and closes in one go:

In [ ]:
print(repr(Path("shopping.txt").read_text()))

## Text in a file is bytes

A file holds **bytes** (numbers from 0 to 255), not characters. `.encode(...)` turns a string into bytes, `.decode(...)` turns bytes back into a string. The rule for how is the **encoding**; the usual one is UTF-8.

**Predict:** how many bytes does each character take?

In [ ]:
for text in ["a", "é", "€", "😀"]:
    data = text.encode("utf-8")
    print(text, len(text), data, len(data))

`b'...'` is a `bytes` value; `\xc3` is one byte, written in hexadecimal. In UTF-8 a character takes 1 to 4 bytes. English letters, digits and punctuation take one, which is why you usually don't notice.

**Predict:**

In [ ]:
data = "Grüße".encode("utf-8")
print(data)
print(data.decode("utf-8"))
print(data.decode("cp1252"))
print(data.decode("ascii"))

Decoding with the wrong encoding gives garbage (`GrÃ¼ÃŸe`, you'll see this on websites and in e-mails) or a `UnicodeDecodeError`. Bytes don't say which encoding they were written in: you have to know.

`open(...)` encodes and decodes for you, with the operating system's default encoding unless you say otherwise. On Linux and macOS that's UTF-8; on Windows it's often `cp1252` (until Python 3.15, which uses UTF-8 everywhere). So a file with umlauts written on one computer can come out garbled on another. Say which encoding you mean:

In [ ]:
with open("shopping.txt", "w", encoding="utf-8") as file:
    file.write("Äpfel\n")

with open("shopping.txt", encoding="utf-8") as file:
    print(file.read())

In [ ]:
with open("shopping.txt", "rb") as file:
    print(file.read())

## Not every file is plain lines: `json` and `csv`

Two more standard library modules, for the two most common text formats for data. Check whether a module exists before writing a parser yourself.

**JSON** stores lists, dicts, strings and numbers as text, and gets them back:

In [ ]:
import json

scores = {"Ada": [17, 20], "Grace": [20]}
text = json.dumps(scores)
print(repr(text))
print(json.loads(text) == scores)

**CSV** ("comma-separated values") is what spreadsheets export. **Predict:** what would `line.split(",")` give for Ada's line?

In [ ]:
import csv

with open("scores.csv", "w") as file:
    file.write('name,points,remark\nAda,17,"late, but fine"\nGrace,20,\n')

with open("scores.csv") as file:
    for row in csv.reader(file):
        print(row)

In [ ]:
with open("scores.csv") as file:
    for line in file:
        print(line.split(","))

Four pieces instead of three: the comma inside the quotes is part of the remark, and `split` doesn't know about quotes. `csv.reader` does. In the data science weeks, `pandas` reads CSV files for you.

In [ ]:
# tidy up the files this notebook wrote
Path("shopping.txt").unlink()
Path("scores.csv").unlink()

# Part 3: Errors

## Reading a traceback

**Predict:** what does this print?

In [ ]:
def average(numbers):
    return sum(numbers) / len(numbers)


def report(scores):
    for name, points in scores.items():
        print(name, average(points))


report({"Ada": [17, 20], "Grace": []})

Ada's line is printed, then the program stops. Read the error message (the **traceback**) from the **bottom** up:

1. The last line: the **type** of the error (`ZeroDivisionError`) and a message.
2. Above it: the line where it happened, in `average`.
3. Further up: who called that, `report`, and who called *that*. It's the call stack from Wednesday's debugger, printed out.

The line where it crashed isn't necessarily the line with the bug. Is `average` wrong, or should `report` never pass it an empty list? That's a decision about what `average` promises, and its docstring should say.

Errors you've met so far:

| Error | Typical cause |
|---|---|
| `NameError` | a name that was never assigned (typo, or a missing `import`) |
| `TypeError` | the wrong kind of value, e.g. `"4000" + 1` |
| `ValueError` | the right kind of value, but not a usable one, e.g. `int("")` |
| `IndexError` | a list position that doesn't exist |
| `KeyError` | a dict key that doesn't exist |
| `ZeroDivisionError` | dividing by zero |

## Catching an error: `try` / `except`

**Predict:** what does this print?

In [ ]:
def to_number(text):
    try:
        return int(text)
    except ValueError:
        return None


print(to_number("42"))
print(to_number("-7"))
print(to_number("forty-two"))

Python runs the `try` block. If a `ValueError` happens inside it, Python jumps straight to the `except ValueError:` block and carries on from there, instead of stopping the program. If nothing goes wrong, the `except` block is skipped.

This is the better way from Part 1: instead of checking first whether `int(...)` will work (`isdigit()`, which gets negative numbers wrong), just do it, and handle the case where it fails.

## `first_number`

**Predict:** what do the two calls print?

In [ ]:
def first_number(texts):
    try:
        return int(texts[0])
    except ValueError:
        return None


print(first_number(["x"]))
print(first_number([]))

`except ValueError` only catches `ValueError`s. The `IndexError` from `texts[0]` on an empty list goes straight through, as if there were no `try` at all. That's good: you planned for "not a number", not for "no list element", so you want to hear about the second one.

You'll see this on the internet:

```python
try:
    ...
except:           # catches everything
    pass          # ...and says nothing
```

It catches **every** error, including your own typos (`NameError`), and throws it away. The program carries on with wrong results, and nothing tells you where it went wrong. Our CI rejects `except:` and `except ...: pass` (that's ruff's E722 and S110). Name the error you expect, and do something sensible with it.

## Raising an error yourself: `raise`

In Exercise 2, `step` met an opcode it didn't know. Across your pull requests it returned `-1`, returned `None`, skipped the instruction, returned a string, or **raised an error**. Here's the last one, for a simpler function:

In [ ]:
def grade(points):
    """Return "pass" for 10 to 20 points, "fail" for 0 to 9.

    Points outside 0 to 20 raise a ValueError.
    """
    if points < 0 or points > 20:
        raise ValueError(f"points must be between 0 and 20, got {points}")
    if points >= 10:
        return "pass"
    return "fail"

**Predict:** what does this print?

In [ ]:
print(grade(15))
print(grade(25))
print(grade(5))

`raise` stops the function immediately, like `return`, but **without** a result: the caller stops too, and the one above it, until someone catches the error, or until the program ends with a traceback. So `grade(5)` never runs.

Why raise instead of returning `-1` or `None`?

- A caller can't mistake an error for a real result. `-1` could be a real result; `None` from `step` means "halt".
- Nobody can forget to check for it: an unhandled error stops the program **at the place where things went wrong**, not three functions later. On Wednesday's debugger demo, a `step` that raised on an unknown opcode would have stopped right at the bug.
- The message says what went wrong, and with which value. Put the value in it, as above.

Return a special value only if it's a normal outcome that callers are expected to deal with. Exercise 3's "`None` if enough space is free already" is one.

## Testing that something raises

A test for "this raises" uses `pytest.raises`, in a `with` block:

In [ ]:
import pytest

with pytest.raises(ValueError):
    grade(25)
print("got the ValueError, as expected")

**Predict:** and this one?

In [ ]:
with pytest.raises(ValueError):
    grade(15)

`pytest.raises` fails if the block **doesn't** raise the error you named, and passes quietly if it does. That's shorter and safer than writing `try`/`except` in a test yourself: a hand-written version easily passes when nothing was raised at all.

In a test file:

```python
def test_grade_too_many_points():
    with pytest.raises(ValueError):
        grade(25)
```

# Part 4: `match`

## Choosing by value

A long `if`/`elif` chain that compares one value with several constants can be written as a `match`. **Predict:**

In [ ]:
def describe(command):
    match command:
        case "start":
            return "starting"
        case "stop" | "halt":
            return "stopping"
        case _:
            return "unknown command: " + command


print(describe("halt"))
print(describe("pause"))

- `match` takes a value; Python tries the `case`s from top to bottom and runs the **first** one that fits.
- `|` means "or": either value fits.
- `_` fits anything. It's the `else` of a `match`, so it goes last.

The same as

```python
if command == "start":
    ...
elif command == "stop" or command == "halt":
    ...
else:
    ...
```

## Matching the shape of a list

`case` can also describe what a list looks like. **Predict:** what does each call return?

In [ ]:
def command(line):
    match line.split():
        case ["cd", ".."]:
            return "up"
        case ["cd", name]:
            return "into " + name
        case ["ls"]:
            return "list"
        case _:
            return "?"


print(command("cd .."))
print(command("cd photos"))
print(command("ls -l"))
print(command("cd"))

A list pattern only fits a list of **exactly** that length. Strings in it must be equal; a **name** in it (`name`) fits anything and is assigned that element, like unpacking. Order matters: `["cd", ".."]` has to come before `["cd", name]`, because `name` would also fit `".."`.

That's Exercise 3's transcript, one `case` per kind of line.

## A name in a `case`

**Predict:** what does this print?

In [ ]:
ADD = 1

match 2:
    case ADD:
        print("add")

A plain name in a `case` doesn't compare, it **assigns**: `ADD` fits anything and becomes `2`. Write the value itself (`case 1:`), or use `if opcode == ADD:` if you want names.

# Part 5: Before Exercise 4

## Trace by hand

Exercise 4 extends your computer: parameters can now be **values** instead of addresses ("immediate mode"), and there are instructions for input and output. The README explains how the digits of an opcode like `1002` say which is which. Trace these on paper, with the README open:

1. `[1002, 4, 3, 4, 33]`: what is memory after the first instruction?
2. `[1101, 100, -1, 4, 0]`: what is memory after the first instruction?
3. `[3, 0, 4, 0, 99]` with input `7`: what's output?
4. `[102, 3, 5, 5, 99, 7]`: what is memory after the first instruction?

## Summary

- `split()` splits at any whitespace and never gives `""`; `split(",")` splits at every comma and keeps empty pieces
- `strip()` removes whitespace at both ends; `splitlines()` splits text into lines
- `repr(s)` shows what's really in a string, `"\n"` included
- `isdigit()` is `False` for `"-42"` and `True` for `"²"`: try `int(...)` instead
- `float` reads more than `int` (`"3.5"`, `"1e3"`, `"inf"`)
- Strings are Unicode: equal-looking text can be different characters (`unicodedata.name`, `unicodedata.normalize`)
- Open files with `with open(...) as file:`, which closes them however the block ends
- `file.read()` gives one string; `for line in file:` gives lines that still end with `"\n"`
- `"w"` empties the file first; `"a"` appends
- Files hold bytes; `encode`/`decode` convert with an encoding. Write `encoding="utf-8"` in `open(...)`
- Bare file names are looked up in the working directory; `Path(__file__).parent / "name"` is next to your code
- Check the standard library before writing a parser: `json`, `csv`
- Read a traceback from the bottom: error type and message, then where, then who called it
- `try`/`except SomeError:` catches only that error; never a bare `except:` or `except ...: pass`
- `raise ValueError(f"... {value}")` when a function can't keep its promise; say so in the docstring
- `with pytest.raises(ValueError):` tests that an error is raised
- `match`/`case`: the first fitting case wins, `|` is "or", `_` fits anything; a list pattern fits that exact length, and a plain name in a pattern assigns rather than compares